# Project: Build a Multi-user Conversational Product Recommendation Agent

![](https://i.imgur.com/7ZHpO6U.png)


___Created By: Khwaja___

In [1]:
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install gdown
!pip install rich

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 72.4 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.24.0
    Uninstalling openai-2.24.0:
      Successfully uninstalled openai-2.24.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.16
    Uninstalling langchain-core-1.2.16:
      Successfully uninstalled langchain-core-1.2.16
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.3

In [2]:
from getpass import getpass
import os

OPENAI_KEY = getpass('Enter Open AI API Key: ')
os.environ['OPENAI_API_KEY'] = OPENAI_KEY

Enter Open AI API Key: ··········


In [3]:
# download it manually from https://drive.google.com/file/d/1tAwsv97fICL74uJH9fDlxNEZ_YFFJS3W/view?usp=sharing
# or use gdown as follows to download it automatically
!gdown 1tAwsv97fICL74uJH9fDlxNEZ_YFFJS3W

Downloading...
From: https://drive.google.com/uc?id=1tAwsv97fICL74uJH9fDlxNEZ_YFFJS3W
To: /content/Ecommerce_Product_List.csv
100% 5.19k/5.19k [00:00<00:00, 15.0MB/s]


In [4]:
import pandas as pd

df = pd.read_csv('./Ecommerce_Product_List.csv')
df.head()

,Product_ID,Product_Name,Category,Price_USD,Rating,Description
0,P001,AlphaBook Pro,Laptop,1200,4.5,The AlphaBook Pro features a 15-inch Retina di...
1,P002,BetaTab S,Tablet,500,4.2,BetaTab S is a lightweight tablet with a 15-in...
2,P003,GammaPhone X,Smartphone,800,4.7,GammaPhone X comes with a 6.7-inch AMOLED disp...
3,P004,DeltaWatch 2,Smartwatch,300,4.0,"DeltaWatch 2 offers fitness tracking, heart ra..."
4,P005,EpsilonCam 300,Camera,600,4.1,EpsilonCam 300 is a mirrorless camera with a 2...


## Product Recommender Agent Workflow

We will:

- Build a Pandas Code Tool Executor to filter products based on category, rating, price
- Build a LLM Recommender Chain to filter products based on natural language descriptions using an LLM
- Build a query rephraser LLM Chain to combine multiple queries in a conversation to generate better queries
- Combine all of these into a single chain
- Add conversational memory to this system

![](https://i.imgur.com/qJIsErH.png)

In [5]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0)

#User provides a query in natrual language that gets passsed to Pandas tool
from langchain_core.runnables import chain

@chain
def pandas_code_tool(query):
  result_df = eval(query)

  if result_df.empty:
    return df.to_markdown()

  else:
    return result_df.to_markdown()

# Now we will create a proimpt for the LLM to convert the natural query to a well fromatted pandas text query which will then be passed to the "pandas_code_tool"
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

FILTER_PROMPT = """Given the following schema of a dataframe table,
            your task is to figure out the best pandas query to
            filter the dataframe based on the user query which
            will be in natural language.

            The schema is as follows:

            #   Column        Non-Null Count  Dtype
            ---  ------        --------------  -----
            0   Product_ID    30 non-null     object
            1   Product_Name  30 non-null     object
            2   Category      30 non-null     object
            3   Price_USD     30 non-null     int64
            4   Rating        30 non-null     float64
            5   Description   30 non-null     object

            Category has values: ['Laptop', 'Tablet', 'Smartphone',
                                  'Smartwatch', 'Camera',
                                  'Headphones', 'Mouse', 'Keyboard',
                                  'Monitor', 'Charger']

            Rating ranges from 1 - 5 in floats

            You will try to figure out the pandas query focusing
            only on Category, Price_USD, and Rating if the user mentions
            anything about these in their natural language query.
            Do not make up column names, only use the above.
            If not the pandas query should just return the full dataframe.
            Remember the dataframe name is df.

            Just return only the pandas query and nothing else.
            Do not return the results as markdown, just return the query

            User Query: {user_query}
            Pandas Query:
        """
FILTER_PROMPT_TEMPLATE = ChatPromptTemplate.from_template(FILTER_PROMPT)

data_filter_chain = FILTER_PROMPT_TEMPLATE | chatgpt | StrOutputParser() | pandas_code_tool

In [6]:
filtered_table = data_filter_chain.invoke({'user_query' : 'I want to buy a laptop'})
print(filtered_table)

|    | Product_ID   | Product_Name   | Category   |   Price_USD |   Rating | Description                                                                                                                                                             |
|---:|:-------------|:---------------|:-----------|------------:|---------:|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
|  0 | P001         | AlphaBook Pro  | Laptop     |        1200 |      4.5 | The AlphaBook Pro features a 15-inch Retina display, 16GB RAM, 512GB SSD, and an Intel Core i7 processor. Ideal for professionals who need performance and portability. |
| 10 | P011         | AlphaBook Air  | Laptop     |        1000 |      4.6 | AlphaBook Air offers a sleek 13-inch Retina display, 8GB RAM, 256GB SSD, and Intel Core i5, designed for portability and efficiency.                                    |
| 20 | P021 

In [7]:
filtered_table = data_filter_chain.invoke({'user_query' : 'I want to buy a laptop with at least 16gb rab'})
print(filtered_table)

|    | Product_ID   | Product_Name   | Category   |   Price_USD |   Rating | Description                                                                                                                                                             |
|---:|:-------------|:---------------|:-----------|------------:|---------:|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
|  0 | P001         | AlphaBook Pro  | Laptop     |        1200 |      4.5 | The AlphaBook Pro features a 15-inch Retina display, 16GB RAM, 512GB SSD, and an Intel Core i7 processor. Ideal for professionals who need performance and portability. |
| 10 | P011         | AlphaBook Air  | Laptop     |        1000 |      4.6 | AlphaBook Air offers a sleek 13-inch Retina display, 8GB RAM, 256GB SSD, and Intel Core i5, designed for portability and efficiency.                                    |
| 20 | P021 

In [8]:
# Now let's apply a filter on the derscription

RECOMMEND_PROMPT = """Act as an expert retail product advisor
                      Given the following table of products,
                      focus on the product attributes and description in the table
                      and based on the user query below do the following

                      - Recommend the most appropriate products based on the query
                      - Recommedation should have product name, price,  rating, description
                      - Also add a brief on why you recommend the product
                      - Do not make up products or recommend products not in the table
                      - If some specifications do not match focus on the ones which match and recommend
                      - If nothing matches recommend 5 random products from the table
                      - Do not generate anything else except the fields mentioned above

                    In case the user query is just a generic query or greeting
                    respond to them appropriately without recommending any products

                    Product Table:
                    {product_table}

                    User Query:
                    {user_query}

                    Recommendation:
                    """
RECOMMEND_PROMPT_TEMPLATE = ChatPromptTemplate.from_template(RECOMMEND_PROMPT)

recommend_chain = RECOMMEND_PROMPT_TEMPLATE | chatgpt | StrOutputParser()

In [9]:
recommed_df = recommend_chain.invoke({
    'user_query': 'I want to buy a laptop with atleast 16gb ram',
    'product_table': filtered_table
})

print(recommed_df)

| Product_ID   | Product_Name   | Category   |   Price_USD |   Rating | Description                                                                                                                                                             |
|:-------------|:---------------|:-----------|------------:|---------:|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| P001         | AlphaBook Pro  | Laptop     |        1200 |      4.5 | The AlphaBook Pro features a 15-inch Retina display, 16GB RAM, 512GB SSD, and an Intel Core i7 processor. Ideal for professionals who need performance and portability. |

I recommend the AlphaBook Pro for your requirement of a laptop with at least 16GB of RAM. It not only meets the RAM specification but also offers a 15-inch Retina display, 512GB SSD, and an Intel Core i7 processor, making it a powerful and efficient choice for professional

In [10]:
response = recommend_chain.invoke({"user_query": """looking for a tablet with greater than 10 inch display
                                                           and at least 64GB storage""",
                                   "product_table": filtered_table})

In [11]:
tab_data = data_filter_chain.invoke({"user_query": "looking for a tablet with greater than 64gb ram"})
print(tab_data)

|    | Product_ID   | Product_Name   | Category   |   Price_USD |   Rating | Description                                                                                                                              |
|---:|:-------------|:---------------|:-----------|------------:|---------:|:-----------------------------------------------------------------------------------------------------------------------------------------|
|  1 | P002         | BetaTab S      | Tablet     |         500 |      4.2 | BetaTab S is a lightweight tablet with a 15-inch display, 4GB RAM, and 64GB storage. Perfect for entertainment and light work on the go. |
| 11 | P012         | BetaTab Lite   | Tablet     |         400 |      4.1 | BetaTab Lite is a budget-friendly 10-inch tablet with 2GB RAM and 32GB storage, perfect for casual browsing and entertainment.           |
| 21 | P022         | SpectraTab Max | Tablet     |         600 |      4.3 | SpectraTab Max features a 12-inch Retina display, 6GB RAM, 

In [12]:
response = recommend_chain.invoke({"user_query": """looking for a tablet with greater than 10 inch display
                                                           and at least 64GB storage""",
                                   "product_table": tab_data})

print(response)

| Product_ID   | Product_Name   | Category   |   Price_USD |   Rating | Description                                                                                                                              |
|:-------------|:---------------|:-----------|------------:|---------:|:-----------------------------------------------------------------------------------------------------------------------------------------|
| P002         | BetaTab S      | Tablet     |         500 |      4.2 | BetaTab S is a lightweight tablet with a 15-inch display, 4GB RAM, and 64GB storage. Perfect for entertainment and light work on the go. |

I recommend the BetaTab S for your requirements as it has a 15-inch display and 64GB storage, meeting your criteria for a tablet with a greater than 10-inch display and at least 64GB storage. It also has a good rating of 4.2, making it a reliable choice for entertainment and work on the go.


In [13]:
from operator import itemgetter

combined_chain = (
    {'user_query' : itemgetter('user_query'),
     'product_table': data_filter_chain

     }
    |

    recommend_chain
)

In [14]:
response = combined_chain.invoke({"user_query": "looking for a cheap laptop in the range of 500 - 1000"})
print(response)

| Product_Name   | Price_USD   | Rating   | Description                                                                                                                          |
|:---------------|------------:|---------:|:-------------------------------------------------------------------------------------------------------------------------------------|
| AlphaBook Air  | 1000        | 4.6      | AlphaBook Air offers a sleek 13-inch Retina display, 8GB RAM, 256GB SSD, and Intel Core i5, designed for portability and efficiency. |

I recommend the AlphaBook Air as it falls within the price range of 500 - 1000 USD and offers great value for the price. It has a high rating of 4.6 and comes with features such as a 13-inch Retina display, 8GB RAM, 256GB SSD, and Intel Core i5 processor, making it a good choice for those looking for a budget-friendly yet efficient laptop.


In [15]:
df[df['Category'] == 'Laptop']

,Product_ID,Product_Name,Category,Price_USD,Rating,Description
0,P001,AlphaBook Pro,Laptop,1200,4.5,The AlphaBook Pro features a 15-inch Retina di...
10,P011,AlphaBook Air,Laptop,1000,4.6,AlphaBook Air offers a sleek 13-inch Retina di...
20,P021,OmniBook Elite,Laptop,1300,4.7,OmniBook Elite is a powerful 14-inch laptop wi...


In [16]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough

# used to retrieve conversation history from database
# based on a specific user or session ID
def get_session_history_db(session_id):
    return SQLChatMessageHistory(session_id, "sqlite:///memory.db")


SYS_PROMPT = """You are a retail product expert.
                Carefully analyze the following conversation history
                and the current user query.
                Refer to the history and rephrase the current user query
                into a standalone query which can be used without the history
                for making search queries.
                Rephrase only if needed.
                Just return the query and do not answer it.
            """

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", SYS_PROMPT),
        MessagesPlaceholder(variable_name="history"),
        ("human", """Current User Query:
                     {human_input}
                  """),
    ]
)

# create a memory buffer window function to return the last K conversations
def memory_buffer_window(messages, k=10): # 10 here means retrieve only last 2*10 user-AI conversations
    return messages[-(2*k):]

# create a basic LLM Chain which only sends the last K conversations per user
rephrase_query_chain = (
    RunnablePassthrough.assign(history=lambda x: memory_buffer_window(x["history"]))
      |
    prompt_template
      |
    chatgpt
      |
    StrOutputParser()
)

In [17]:
combined_chain = (
         {
             'human_input' : itemgetter('human_input'),
             'history' : itemgetter('history')
         }
           |
        {
            'user_query': rephrase_query_chain
        }
           |
        RunnablePassthrough.assign(product_table=data_filter_chain)
            |
        recommend_chain
)

In [18]:
from rich.console import Console
from rich.markdown import Markdown
from langchain_core.runnables import RunnableWithMessageHistory
# create a conversation chain which can load memory based on specific user or session id
conv_chain = RunnableWithMessageHistory(
    combined_chain,
    get_session_history_db,
    input_messages_key="human_input",
    history_messages_key="history",
)


# create a utility function to take in current user input prompt and their session ID
# streams result live back to the user from the LLM
def chat_with_llm(prompt: str, session_id: str):
    response = conv_chain.invoke({"human_input": prompt},
                                 {'configurable': { 'session_id': session_id}})
    console = Console()
    console.print(Markdown(response))

In [19]:
user_id = 'jim001'
prompt = "looking for a tablet"
chat_with_llm(prompt, user_id)

/usr/local/lib/python3.12/dist-packages/langchain_core/runnables/history.py:596: LangChainDeprecationWarning: `connection_string` was deprecated in LangChain 0.2.2 and will be removed in 1.0. Use connection instead.
  message_history = self.get_session_history(


 1 Product Name: SpectraTab Max Price: $600 Rating: 4.3 Description: SpectraTab Max features a 12-inch Retina      
   display, 6GB RAM, and 128GB storage, designed for entertainment and professional use. Reason for recommendation:
   SpectraTab Max is a popular tablet with high-end features suitable for both entertainment and professional use, 
   making it a top choice in the market.                                                                           
 2 Product Name: BetaTab S Price: $500 Rating: 4.2 Description: BetaTab S is a lightweight tablet with a 15-inch   
   display, 4GB RAM, and 64GB storage. Perfect for entertainment and light work on the go. Reason for              
   recommendation: BetaTab S offers a balance between performance and portability, making it a popular choice among
   users looking for a versatile tablet.                                                                           
 3 Product Name: BetaTab Lite Price: $400 Rating: 4.1 Description: BetaTab Lite is a budget-friendly 10-inch tablet
   with 2GB RAM and 32GB storage, perfect for casual browsing and entertainment. Reason for recommendation: BetaTab
   Lite is a popular choice for users on a budget who still want a reliable tablet for everyday use.               

In this case, the recommended tablets are based on popularity, features, and price range, making them suitable     
options for users looking for popular tablets in the market.